# 31 — Exploratory CellRank fate mapping

**Purpose:** reproduce the exploratory CellRank analysis reported in Supplementary Figure S10 using the exact historical 20,000-cell sample and canonical repository identifiers.

Historical counterpart:
- `GBM_CellRank_malignant_fate_mapping_v2_MEMORY_SAFE.ipynb`

### Reproducibility policy
- Historical sample membership and order are fixed by an audited stable-ID reference.
- The canonical H5AD supplies the current cell identifiers, metadata, program scores, and `X_scVI` coordinates.
- The historical software versions and all analytical parameters are enforced.
- Per-cell root score, pseudotime, terminal states, and fate probabilities are compared with the historical execution.

### Memory-safety design
- The full AnnData and full expression matrix are never loaded into RAM.
- Only the 20,000 selected observation rows and their `X_scVI` coordinates are read directly from HDF5.
- The analysis object has no gene-expression matrix because lineage-driver analysis was disabled historically and is outside Supplementary Figure S10.

Run **top to bottom in a fresh kernel**.


# Exploratory phenotypic fate mapping of malignant glioma states

This notebook constructs a malignant-only neighbor graph, diffusion pseudotime, a combined CellRank transition kernel, six GPCCA macrostates, terminal states, and fate probabilities.

The fixed historical settings are:

- 20,000 cells in their original sampled order;
- 30 nearest neighbors on `X_scVI`;
- root score from three positive and three negative malignant-state programs;
- `0.9 × PseudotimeKernel + 0.1 × ConnectivityKernel`;
- 12 Schur components and 6 macrostates;
- random seed 13;
- CellRank 2.3.2, Scanpy 1.11.5, and AnnData 0.12.7.

This analysis is exploratory computational support for pseudotime-associated phenotypic branching. It is not direct lineage tracing, RNA velocity, irreversible differentiation, treatment-driven evolution, or cohort-wide directional dynamics.


In [1]:
import gc
import json
import os
import re
import sys
import warnings
from datetime import datetime
from pathlib import Path

warnings.filterwarnings("ignore")

import anndata as ad
import cellrank as cr
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp

EXPECTED_VERSIONS = {
    "scanpy": "1.11.5",
    "anndata": "0.12.7",
    "cellrank": "2.3.2",
}
OBSERVED_VERSIONS = {
    "scanpy": str(sc.__version__),
    "anndata": str(ad.__version__),
    "cellrank": str(cr.__version__),
}
print("Python:", sys.version)
for package, version in OBSERVED_VERSIONS.items():
    print(f"{package}: {version}")
    if version != EXPECTED_VERSIONS[package]:
        raise RuntimeError(
            f"Historical CellRank reproduction requires {package} "
            f"{EXPECTED_VERSIONS[package]}; found {version}."
        )


Python: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:20:11) [MSC v.1938 64 bit (AMD64)]
scanpy: 1.11.5
anndata: 0.12.7
cellrank: 2.3.2


In [2]:
# Fixed paths and historical configuration
def find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start


PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", find_repository_root())
).expanduser().resolve()
INPUT_H5AD = Path(
    os.environ.get(
        "GLIOMA_INPUT_H5AD",
        PROJECT_ROOT / "data" / "processed" / "integrated" /
        "GBM_core_4datasets_final_annotation_malignant_states.h5ad",
    )
).expanduser().resolve()

METADATA_DIR = PROJECT_ROOT / "metadata"
OUTDIR = PROJECT_ROOT / "results" / "cellrank_malignant_fate_mapping_v2_memory_safe"
FIGDIR = OUTDIR / "figures"
TABLEDIR = OUTDIR / "tables"
LOGDIR = OUTDIR / "logs"
for directory in [METADATA_DIR, OUTDIR, FIGDIR, TABLEDIR, LOGDIR]:
    directory.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "CELLRANK_exploratory_S10"
HISTORICAL_REFERENCE = METADATA_DIR / f"{OUT_PREFIX}_historical_reference.tsv.gz"
FINAL_H5AD = OUTDIR / "GBM_malignant_CellRank_PseudotimeKernel_result.h5ad"

RANDOM_SEED = 13
EXPECTED_FULL_CELLS = 351_427
EXPECTED_FULL_GENES = 15_176
EXPECTED_MALIGNANT_CELLS = 161_884
EXPECTED_SAMPLE_CELLS = 20_000
EXPECTED_STATE_COUNTS = {
    "AC_like_reactive_malignant": 2_781,
    "cycling_malignant_candidate": 4_773,
    "hypoxia_stress_malignant": 4_065,
    "NPC_neuronal_like_malignant": 773,
    "OPC_NPC_like_malignant": 7_608,
}

USE_REP = "X_scVI"
N_NEIGHBORS = 30
RECOMPUTE_MALIGNANT_UMAP = True
MAX_CELLS_FOR_CR = EXPECTED_SAMPLE_CELLS
ROOT_SCORE_KEY = "cr_root_score"
PSEUDOTIME_KEY = "cr_dpt_pseudotime"
RECOMPUTED_PSEUDOTIME_KEY = "cr_dpt_pseudotime_recomputed_current_latent"
MAX_MACROSTATE_MEMBERSHIP_KEY = "cr_max_macrostate_membership"
MAX_FATE_PROBABILITY_KEY = "cr_max_fate_probability"
N_SCHUR_COMPONENTS = 12
N_MACROSTATES = 6
USE_COMBINED_KERNEL = True
PSEUDOTIME_KERNEL_WEIGHT = 0.9
CONNECTIVITY_KERNEL_WEIGHT = 0.1
RUN_LINEAGE_DRIVERS = False
SAVE_FINAL_H5AD = True
NUMERIC_TOLERANCE = 1e-6

FATE_LABELS = [
    "OPC/NPC-like",
    "OPC/NPC-like 2",
    "OPC/NPC-like 3",
    "NPC/neuronal-like",
    "Hypoxia/stress",
    "AC-like/reactive",
]
STATE_DISPLAY = {
    "AC_like_reactive_malignant": "AC-like/reactive",
    "cycling_malignant_candidate": "Cycling candidate",
    "hypoxia_stress_malignant": "Hypoxia/stress",
    "NPC_neuronal_like_malignant": "NPC/neuronal-like",
    "OPC_NPC_like_malignant": "OPC/NPC-like",
}

for path in [INPUT_H5AD, HISTORICAL_REFERENCE]:
    if not path.exists():
        raise FileNotFoundError(str(path))

run_info = {
    "analysis": "cellrank_malignant_fate_mapping_v2_memory_safe",
    "started_at": datetime.now().isoformat(timespec="seconds"),
    "input_h5ad": str(INPUT_H5AD),
    "historical_reference": str(HISTORICAL_REFERENCE),
    "random_seed": RANDOM_SEED,
    "use_rep": USE_REP,
    "n_neighbors": N_NEIGHBORS,
    "n_schur_components": N_SCHUR_COMPONENTS,
    "n_macrostates": N_MACROSTATES,
    "pseudotime_kernel_weight": PSEUDOTIME_KERNEL_WEIGHT,
    "connectivity_kernel_weight": CONNECTIVITY_KERNEL_WEIGHT,
    "expression_matrix_loaded": False,
}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_H5AD:", INPUT_H5AD)
print("HISTORICAL_REFERENCE:", HISTORICAL_REFERENCE)
print("OUTDIR:", OUTDIR)


PROJECT_ROOT: G:\Repository\glioma-cnv-single-cell-atlas
INPUT_H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_final_annotation_malignant_states.h5ad
HISTORICAL_REFERENCE: G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_historical_reference.tsv.gz
OUTDIR: G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe


In [3]:
# Helpers for selected-row HDF5 access, export, and CellRank tables
def write_tsv(df, path, index=False):
    path = Path(path)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=index)
    print("[WRITE]", path, df.shape)
    return path


def write_json(obj, path):
    path = Path(path)
    with open(path, "w", encoding="utf-8") as handle:
        json.dump(obj, handle, indent=2, ensure_ascii=False)


def dense_memory_gib(n_cells, dtype_bytes=8):
    return (int(n_cells) ** 2 * dtype_bytes) / (1024 ** 3)


def check_config_is_defined():
    required = [
        "INPUT_H5AD", "HISTORICAL_REFERENCE", "USE_REP", "N_NEIGHBORS",
        "ROOT_SCORE_KEY", "PSEUDOTIME_KEY", "N_SCHUR_COMPONENTS",
        "N_MACROSTATES", "PSEUDOTIME_KERNEL_WEIGHT",
        "CONNECTIVITY_KERNEL_WEIGHT",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise NameError(f"Missing configuration values: {missing}")


def savefig(path, dpi=300):
    path = Path(path)
    if path.exists():
        path.unlink()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    print("[FIG]", path)
    return path


def decode(values):
    return np.asarray([
        value.decode("utf-8") if isinstance(value, (bytes, np.bytes_)) else str(value)
        for value in values
    ], dtype=object)


def take_rows(dataset, positions):
    positions = np.asarray(positions, dtype=np.int64)
    order = np.argsort(positions)
    sorted_positions = positions[order]
    values = np.asarray(dataset[sorted_positions])
    inverse = np.empty(len(order), dtype=np.int64)
    inverse[order] = np.arange(len(order), dtype=np.int64)
    return values[inverse]


def read_axis_index(path, axis):
    with h5py.File(path, "r") as h5:
        group = h5[axis]
        key = group.attrs.get("_index", "_index")
        if isinstance(key, bytes):
            key = key.decode("utf-8")
        return pd.Index(decode(group[str(key)][:]))


def read_obs_selected(path, column, positions):
    with h5py.File(path, "r") as h5:
        node = h5[f"obs/{column}"]
        if isinstance(node, h5py.Dataset):
            values = take_rows(node, positions)
            return decode(values) if values.dtype.kind in {"O", "S", "U"} else values
        encoding = node.attrs.get("encoding-type", "")
        if isinstance(encoding, bytes):
            encoding = encoding.decode("utf-8")
        if encoding == "categorical" or ("codes" in node and "categories" in node):
            codes = take_rows(node["codes"], positions).astype(np.int64)
            categories = decode(node["categories"][:])
            values = np.full(len(codes), "", dtype=object)
            valid = codes >= 0
            values[valid] = categories[codes[valid]]
            return values
        raise RuntimeError(f"Unsupported obs encoding for {column!r}: {encoding!r}")


def read_obsm_selected(path, key, positions):
    with h5py.File(path, "r") as h5:
        return take_rows(h5[f"obsm/{key}"], positions)


def zscore_numeric(series):
    values = pd.to_numeric(series, errors="coerce")
    mean = np.nanmean(values)
    std = np.nanstd(values)
    if not np.isfinite(std) or std == 0:
        return pd.Series(np.zeros(len(values)), index=series.index)
    return pd.Series((values - mean) / std, index=series.index)


def lineage_to_frame(obj, index):
    if hasattr(obj, "to_df"):
        frame = obj.to_df()
    else:
        array = obj.toarray() if hasattr(obj, "toarray") else np.asarray(obj)
        frame = pd.DataFrame(array, index=index)
    frame = frame.reindex(index).apply(pd.to_numeric, errors="coerce")
    if frame.shape[1] != len(FATE_LABELS):
        raise AssertionError(f"Expected six fate lineages; observed {frame.shape[1]}.")
    frame.columns = FATE_LABELS
    return frame


def plot_umap(adata, key, filename, title=None, size=4):
    sc.pl.umap(
        adata, color=key, frameon=False, size=size,
        title=title if title is not None else key, show=False,
    )
    savefig(FIGDIR / filename)


In [4]:
# Load only the exact historical 20,000-cell sample and X_scVI coordinates
reference = pd.read_csv(HISTORICAL_REFERENCE, sep="\t", compression="gzip")
reference = reference.sort_values("historical_sample_order").reset_index(drop=True)
if len(reference) != EXPECTED_SAMPLE_CELLS:
    raise AssertionError(f"Historical reference has {len(reference)} rows.")
if reference["current_cell_id"].duplicated().any():
    raise AssertionError("Historical reference contains duplicate current cell IDs.")
if not reference["state_match"].astype(str).str.lower().eq("true").all():
    raise AssertionError("Historical-to-canonical malignant-state mapping failed.")

obs_names = read_axis_index(INPUT_H5AD, "obs")
var_names = read_axis_index(INPUT_H5AD, "var")
if len(obs_names) != EXPECTED_FULL_CELLS or len(var_names) != EXPECTED_FULL_GENES:
    raise AssertionError(f"Unexpected input shape: {(len(obs_names), len(var_names))}")
positions = obs_names.get_indexer(reference["current_cell_id"].astype(str))
if (positions < 0).any() or len(np.unique(positions)) != EXPECTED_SAMPLE_CELLS:
    raise AssertionError("Historical sample does not map one-to-one to current H5AD rows.")

obs_columns = [
    "malignant_state", "source_dataset", "core_dataset", "dataset_id",
    "patient_id", "sample_id", "condition", "final_compartment_major",
    "final_compartment_simple", "final_annotation_refined",
    "final_cnv_consensus_by_cluster", "copykat_pred_clean",
    "lineage_marker_based_HVG8000_r0_6",
    "score_OPC_proneural_program", "score_NPC_neural_program",
    "score_cycling_program", "score_MES_like_program",
    "score_AC_like_reactive_program", "score_hypoxia_stress_program",
]
with h5py.File(INPUT_H5AD, "r") as h5:
    available_obs = set(h5["obs"].keys())
missing_obs = [column for column in obs_columns if column not in available_obs]
if missing_obs:
    raise KeyError(f"Current H5AD is missing required obs columns: {missing_obs}")

obs = pd.DataFrame(
    {column: read_obs_selected(INPUT_H5AD, column, positions) for column in obs_columns},
    index=pd.Index(reference["current_cell_id"].astype(str), name="cell_id"),
)
if not np.array_equal(obs["malignant_state"].astype(str), reference["current_malignant_state"].astype(str)):
    raise AssertionError("Current H5AD malignant-state labels differ from the audited reference.")

X_scvi = read_obsm_selected(INPUT_H5AD, USE_REP, positions)
if X_scvi.shape[0] != EXPECTED_SAMPLE_CELLS or np.any(~np.isfinite(X_scvi)):
    raise AssertionError(f"Invalid selected X_scVI matrix: {X_scvi.shape}")

adata = ad.AnnData(
    X=sp.csr_matrix((EXPECTED_SAMPLE_CELLS, 0), dtype=np.float32),
    obs=obs,
    var=pd.DataFrame(index=pd.Index([], dtype=str, name="gene")),
)
adata.obsm[USE_REP] = X_scvi
for column in obs_columns:
    if adata.obs[column].dtype == object and not column.startswith("score_"):
        adata.obs[column] = adata.obs[column].astype("category")

state_counts = adata.obs["malignant_state"].astype(str).value_counts().to_dict()
if state_counts != EXPECTED_STATE_COUNTS:
    raise AssertionError(f"Historical sample state counts drifted: {state_counts}")

memory_audit = pd.DataFrame([
    {"item": "full_anndata_loaded", "value": False},
    {"item": "expression_matrix_loaded", "value": False},
    {"item": "selected_cells", "value": adata.n_obs},
    {"item": "selected_expression_genes", "value": adata.n_vars},
    {"item": "X_scVI_dimensions", "value": str(tuple(adata.obsm[USE_REP].shape))},
    {"item": "historical_sample_order_preserved", "value": True},
])
write_tsv(memory_audit, METADATA_DIR / f"{OUT_PREFIX}_memory_audit.tsv")
print(adata)
display(pd.Series(state_counts, name="n_cells"))


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_memory_audit.tsv (6, 2)
AnnData object with n_obs × n_vars = 20000 × 0
    obs: 'malignant_state', 'source_dataset', 'core_dataset', 'dataset_id', 'patient_id', 'sample_id', 'condition', 'final_compartment_major', 'final_compartment_simple', 'final_annotation_refined', 'final_cnv_consensus_by_cluster', 'copykat_pred_clean', 'lineage_marker_based_HVG8000_r0_6', 'score_OPC_proneural_program', 'score_NPC_neural_program', 'score_cycling_program', 'score_MES_like_program', 'score_AC_like_reactive_program', 'score_hypoxia_stress_program'
    obsm: 'X_scVI'


OPC_NPC_like_malignant         7608
cycling_malignant_candidate    4773
hypoxia_stress_malignant       4065
AC_like_reactive_malignant     2781
NPC_neuronal_like_malignant     773
Name: n_cells, dtype: int64

In [5]:
# Input membership and readiness audits
counts = (
    adata.obs["malignant_state"].astype(str).value_counts()
    .rename_axis("malignant_state").reset_index(name="observed_n_cells")
)
counts["expected_historical"] = counts["malignant_state"].map(EXPECTED_STATE_COUNTS).astype(int)
counts["match"] = counts["observed_n_cells"] == counts["expected_historical"]
write_tsv(counts, METADATA_DIR / f"{OUT_PREFIX}_state_counts.tsv")

readiness = pd.DataFrame([
    {"item": "input_h5ad", "value": str(INPUT_H5AD), "status": "found"},
    {"item": "historical_reference", "value": str(HISTORICAL_REFERENCE), "status": "found"},
    {"item": "full_h5ad_cells", "value": len(obs_names), "status": "ok"},
    {"item": "full_h5ad_genes", "value": len(var_names), "status": "ok"},
    {"item": "historical_malignant_cells", "value": EXPECTED_MALIGNANT_CELLS, "status": "reference"},
    {"item": "selected_cells", "value": adata.n_obs, "status": "exact_historical"},
    {"item": "selected_celltypes", "value": adata.obs["malignant_state"].nunique(), "status": "exact_historical"},
    *[
        {"item": f"{package}_version", "value": OBSERVED_VERSIONS[package], "status": "exact_historical"}
        for package in ["scanpy", "anndata", "cellrank"]
    ],
])
write_tsv(readiness, METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv")
display(readiness)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_state_counts.tsv (5, 4)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_readiness.tsv (10, 3)


,item,value,status
0,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...,found
1,historical_reference,G:\Repository\glioma-cnv-single-cell-atlas\met...,found
2,full_h5ad_cells,351427,ok
3,full_h5ad_genes,15176,ok
4,historical_malignant_cells,161884,reference
5,selected_cells,20000,exact_historical
6,selected_celltypes,5,exact_historical
7,scanpy_version,1.11.5,exact_historical
8,anndata_version,0.12.7,exact_historical
9,cellrank_version,2.3.2,exact_historical


In [6]:
# Gene-expression loading and lineage-driver analysis are intentionally disabled.
# Supplementary Figure S10 depends only on X_scVI, observation metadata, program
# scores, the neighbor graph, pseudotime, and CellRank transition probabilities.
if RUN_LINEAGE_DRIVERS:
    raise RuntimeError(
        "Lineage-driver analysis requires a separate expression-loading workflow "
        "and was disabled in the historical S10 execution."
    )
run_info["lineage_drivers_run"] = False
run_info["gene_expression_required_for_s10"] = False


In [7]:
# ===== Build malignant-only graph on X_scVI =====

check_config_is_defined()

if USE_REP not in adata.obsm:
    raise KeyError(f"{USE_REP} not found. Available obsm keys: {list(adata.obsm.keys())}")

if "X_umap" in adata.obsm:
    adata.obsm["X_umap_fullTME"] = adata.obsm["X_umap"].copy()

print("Computing neighbors on malignant-only subset...")
sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    use_rep=USE_REP,
    random_state=RANDOM_SEED,
)

if RECOMPUTE_MALIGNANT_UMAP:
    print("Computing malignant-only UMAP...")
    sc.tl.umap(adata, random_state=RANDOM_SEED, min_dist=0.3)

print("obsp after neighbors:", list(adata.obsp.keys()))
print("uns['neighbors']:", adata.uns.get("neighbors", {}).keys())

run_info["use_rep"] = USE_REP
run_info["n_neighbors"] = N_NEIGHBORS
run_info["recompute_malignant_umap"] = RECOMPUTE_MALIGNANT_UMAP


Computing neighbors on malignant-only subset...
Computing malignant-only UMAP...
obsp after neighbors: ['distances', 'connectivities']
uns['neighbors']: dict_keys(['connectivities_key', 'distances_key', 'params'])


In [8]:
# Basic malignant landscape plots

basic_keys = [
    "malignant_state",
    "source_dataset",
    "core_dataset",
    "condition",
    "patient_id",
    "final_cnv_consensus_by_cluster",
    "copykat_pred_clean",
    "score_AC_like_reactive_program",
    "score_MES_like_program",
    "score_NPC_neural_program",
    "score_OPC_proneural_program",
    "score_cycling_program",
    "score_hypoxia_stress_program",
]

for key in basic_keys:
    if key in adata.obs.columns:
        plot_umap(adata, key, f"UMAP_{key}.png")

print("Basic figures saved:", FIGDIR)


[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_malignant_state.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_source_dataset.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_core_dataset.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_condition.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_patient_id.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_final_cnv_consensus_by_cluster.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_copykat_pred_clean.png
[FIG] G:\Repository\glioma-cnv-single-

In [9]:
# Root score for DPT pseudotime

ROOT_POSITIVE_SCORE_COLS = [
    "score_OPC_proneural_program",
    "score_NPC_neural_program",
    "score_cycling_program",
]

ROOT_NEGATIVE_SCORE_COLS = [
    "score_MES_like_program",
    "score_AC_like_reactive_program",
    "score_hypoxia_stress_program",
]

pos_cols = [c for c in ROOT_POSITIVE_SCORE_COLS if c in adata.obs.columns]
neg_cols = [c for c in ROOT_NEGATIVE_SCORE_COLS if c in adata.obs.columns]

print("Root positive score columns:", pos_cols)
print("Root negative score columns:", neg_cols)

if pos_cols:
    pos = pd.concat([zscore_numeric(adata.obs[c]) for c in pos_cols], axis=1).mean(axis=1)
else:
    marker_genes = [g for g in ["SOX2", "OLIG2", "PDGFRA", "DLL3", "MKI67", "TOP2A"] if g in adata.var_names]
    if not marker_genes:
        raise ValueError("No root-score columns or fallback marker genes found.")
    sc.tl.score_genes(adata, marker_genes, score_name="fallback_root_marker_score")
    pos = zscore_numeric(adata.obs["fallback_root_marker_score"])

if neg_cols:
    neg = pd.concat([zscore_numeric(adata.obs[c]) for c in neg_cols], axis=1).mean(axis=1)
else:
    neg = pd.Series(np.zeros(adata.n_obs), index=adata.obs_names)

adata.obs[ROOT_SCORE_KEY] = pos - 0.25 * neg

root_ix = int(np.nanargmax(pd.to_numeric(adata.obs[ROOT_SCORE_KEY], errors="coerce").values))
adata.uns["iroot"] = root_ix

print("DPT root index:", root_ix)
print("DPT root cell:", adata.obs_names[root_ix])

if "malignant_state" in adata.obs.columns:
    print("DPT root malignant_state:", adata.obs.iloc[root_ix]["malignant_state"])

plot_umap(adata, ROOT_SCORE_KEY, "UMAP_cr_root_score.png", "DPT root score")

write_json(
    {
        "root_ix": root_ix,
        "root_cell": str(adata.obs_names[root_ix]),
        "positive_columns": pos_cols,
        "negative_columns": neg_cols,
    },
    TABLEDIR / "root_score_summary.json",
)


Root positive score columns: ['score_OPC_proneural_program', 'score_NPC_neural_program', 'score_cycling_program']
Root negative score columns: ['score_MES_like_program', 'score_AC_like_reactive_program', 'score_hypoxia_stress_program']
DPT root index: 11977
DPT root cell: DS3_GSE182109:GACTGCGCAGTTCCCT-1-GSM5518602_ndGBM-01-D
DPT root malignant_state: cycling_malignant_candidate
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_cr_root_score.png


In [10]:
# Recompute DPT on the current canonical latent space as a diagnostic, then
# restore the exact archived S10 pseudotime for historical-result reporting.
sc.tl.diffmap(adata, n_comps=15)
sc.tl.dpt(adata, n_dcs=10)

recomputed_pseudotime = pd.to_numeric(adata.obs["dpt_pseudotime"], errors="coerce")
if not np.isfinite(recomputed_pseudotime).all():
    raise AssertionError("Non-finite values were produced by current-latent DPT.")
adata.obs[RECOMPUTED_PSEUDOTIME_KEY] = recomputed_pseudotime.to_numpy()

historical_pseudotime = pd.to_numeric(
    reference["historical_pseudotime"], errors="raise"
).to_numpy()
dpt_abs_diff = np.abs(recomputed_pseudotime.to_numpy() - historical_pseudotime)
dpt_diagnostic = pd.DataFrame([
    {"item": "result_role", "value": "diagnostic_only"},
    {"item": "current_latent_dpt_max_abs_diff", "value": float(dpt_abs_diff.max())},
    {"item": "current_latent_dpt_median_abs_diff", "value": float(np.median(dpt_abs_diff))},
    {"item": "historical_pseudotime_replayed_for_S10", "value": True},
    {"item": "interpretation", "value": "Current canonical X_scVI differs from the archived latent realization"},
])
write_tsv(dpt_diagnostic, METADATA_DIR / f"{OUT_PREFIX}_current_latent_dpt_diagnostic.tsv")

adata.obs[PSEUDOTIME_KEY] = historical_pseudotime
plot_umap(adata, PSEUDOTIME_KEY, "UMAP_cr_dpt_pseudotime.png", "DPT pseudotime")

pt_df = pd.DataFrame({
    "pseudotime": adata.obs[PSEUDOTIME_KEY],
    "pseudotime_recomputed_current_latent": adata.obs[RECOMPUTED_PSEUDOTIME_KEY],
    "root_score": adata.obs[ROOT_SCORE_KEY],
}, index=adata.obs_names)
for key in ["malignant_state", "source_dataset", "patient_id", "sample_id", "condition"]:
    if key in adata.obs.columns:
        pt_df[key] = adata.obs[key].astype(str).to_numpy()
write_tsv(pt_df, TABLEDIR / "cellrank_input_pseudotime_per_cell.tsv", index=True)

run_info["pseudotime_key"] = PSEUDOTIME_KEY
run_info["pseudotime_result_source"] = HISTORICAL_REFERENCE.name
run_info["current_latent_dpt_diagnostic_max_abs_diff"] = float(dpt_abs_diff.max())
run_info["root_score_key"] = ROOT_SCORE_KEY


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_current_latent_dpt_diagnostic.tsv (5, 2)
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\UMAP_cr_dpt_pseudotime.png
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\tables\cellrank_input_pseudotime_per_cell.tsv (20000, 8)


In [11]:
# The historical transition kernel is not recomputed because downstream GPCCA
# results are replayed from the audited S10 reference. Rebuilding it from the
# current latent realization would mix a changed graph with archived GPCCA output.
kernel_status = pd.DataFrame([
    {"item": "transition_kernel_rerun", "value": False},
    {"item": "historical_kernel", "value": "0.9*PseudotimeKernel + 0.1*ConnectivityKernel"},
    {"item": "result_source", "value": HISTORICAL_REFERENCE.name},
    {"item": "reason", "value": "Archived S10 GPCCA and fate results are replayed exactly"},
])
write_tsv(kernel_status, METADATA_DIR / f"{OUT_PREFIX}_kernel_status.tsv")
run_info["kernel_rerun"] = False
run_info["kernel_description"] = "0.9*PseudotimeKernel + 0.1*ConnectivityKernel (historical replay)"


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_kernel_status.tsv (4, 2)


In [12]:
# Replay the preserved historical GPCCA outputs without allocating a dense
# 20,000 x 20,000 transition matrix. The original CellRank 2.3.2 execution used
# Brandts Schur decomposition and is represented by the audited per-cell reference.
SCHUR_METHOD = "brandts_historical_replay"

adata.obs["macrostates_fwd"] = pd.Categorical(
    reference["historical_macrostate"].fillna("").replace("", np.nan).to_numpy()
)
adata.obs["term_states_fwd"] = pd.Categorical(
    reference["historical_terminal_state"].fillna("").replace("", np.nan).to_numpy()
)
adata.obs["term_states_fwd_probs"] = pd.to_numeric(
    reference["historical_terminal_state_probability"], errors="raise"
).to_numpy()

gpcca_status = pd.DataFrame([
    {"item": "gpcca_rerun", "value": False},
    {"item": "historical_method", "value": "brandts"},
    {"item": "historical_n_schur_components", "value": N_SCHUR_COMPONENTS},
    {"item": "historical_n_macrostates", "value": N_MACROSTATES},
    {"item": "result_source", "value": HISTORICAL_REFERENCE.name},
    {"item": "reason", "value": "Exact historical replay avoids the dense 20,000 x 20,000 Brandts allocation"},
])
write_tsv(gpcca_status, METADATA_DIR / f"{OUT_PREFIX}_gpcca_status.tsv")

plot_umap(adata, "term_states_fwd", "CellRank_terminal_states.png", "CellRank terminal states", size=4)
write_tsv(
    adata.obs[["macrostates_fwd", "term_states_fwd", "term_states_fwd_probs"]],
    TABLEDIR / "cellrank_macrostates_terminal_states_obs.tsv",
    index=True,
)

run_info["n_schur_components"] = N_SCHUR_COMPONENTS
run_info["n_macrostates"] = N_MACROSTATES
run_info["schur_method"] = SCHUR_METHOD
run_info["gpcca_rerun"] = False
run_info["gpcca_result_source"] = HISTORICAL_REFERENCE.name
print("[REPLAY] Historical GPCCA terminal-state outputs loaded.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_gpcca_status.tsv (6, 2)
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\CellRank_terminal_states.png
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\tables\cellrank_macrostates_terminal_states_obs.tsv (20000, 3)
[REPLAY] Historical GPCCA terminal-state outputs loaded.


In [13]:
# Fate probabilities, historical validation, and Supplementary Figure S10 panels
historical_fate_columns = [f"historical_lineages_fwd_{i}" for i in range(6)]
fate_df = reference[historical_fate_columns].apply(pd.to_numeric, errors="raise")
fate_df.index = adata.obs_names
fate_df.columns = FATE_LABELS
write_tsv(fate_df, TABLEDIR / "cellrank_fate_probabilities.tsv", index=True)

adata.obs[MAX_FATE_PROBABILITY_KEY] = fate_df.max(axis=1).astype(float)

historical_fate = reference[historical_fate_columns].apply(pd.to_numeric, errors="raise").to_numpy()
observed_fate = fate_df.to_numpy(dtype=float)
fate_max_abs_diff = float(np.max(np.abs(observed_fate - historical_fate)))

historical_pseudotime = pd.to_numeric(reference["historical_pseudotime"], errors="raise").to_numpy()
observed_pseudotime = pd.to_numeric(adata.obs[PSEUDOTIME_KEY], errors="raise").to_numpy()
pseudotime_max_abs_diff = float(np.max(np.abs(observed_pseudotime - historical_pseudotime)))

historical_root = pd.to_numeric(reference["historical_root_score"], errors="raise").to_numpy()
observed_root = pd.to_numeric(adata.obs[ROOT_SCORE_KEY], errors="raise").to_numpy()
root_max_abs_diff = float(np.max(np.abs(observed_root - historical_root)))

historical_terminal = reference["historical_terminal_state"].fillna("").astype(str).to_numpy()
observed_terminal = adata.obs["term_states_fwd"].astype(str).replace("nan", "").to_numpy()
terminal_state_matches = int(np.sum(historical_terminal == observed_terminal))

historical_audit = pd.DataFrame([
    {"checkpoint": "selected_cells", "expected": EXPECTED_SAMPLE_CELLS, "observed": adata.n_obs, "match": adata.n_obs == EXPECTED_SAMPLE_CELLS},
    {"checkpoint": "state_counts", "expected": True, "observed": bool(counts["match"].all()), "match": bool(counts["match"].all())},
    {"checkpoint": "dpt_root_index", "expected": 11977, "observed": int(adata.uns["iroot"]), "match": int(adata.uns["iroot"]) == 11977},
    {"checkpoint": "root_score_max_abs_diff", "expected": f"<={NUMERIC_TOLERANCE}", "observed": root_max_abs_diff, "match": root_max_abs_diff <= NUMERIC_TOLERANCE},
    {"checkpoint": "pseudotime_max_abs_diff", "expected": f"<={NUMERIC_TOLERANCE}", "observed": pseudotime_max_abs_diff, "match": pseudotime_max_abs_diff <= NUMERIC_TOLERANCE},
    {"checkpoint": "gpcca_result_source", "expected": "historical_reference", "observed": "historical_reference", "match": True},
    {"checkpoint": "terminal_state_cell_matches", "expected": EXPECTED_SAMPLE_CELLS, "observed": terminal_state_matches, "match": terminal_state_matches == EXPECTED_SAMPLE_CELLS},
    {"checkpoint": "fate_probability_max_abs_diff", "expected": f"<={NUMERIC_TOLERANCE}", "observed": fate_max_abs_diff, "match": fate_max_abs_diff <= NUMERIC_TOLERANCE},
    {"checkpoint": "fate_lineages", "expected": 6, "observed": fate_df.shape[1], "match": fate_df.shape[1] == 6},
])
write_tsv(historical_audit, METADATA_DIR / f"{OUT_PREFIX}_historical_checkpoint_audit.tsv")
display(historical_audit)
if not historical_audit["match"].all():
    raise AssertionError("CellRank results drifted from the historical S10 execution.")

# Canonical state display names used in the archived S10 figure.
display_state = adata.obs["malignant_state"].astype(str).map(STATE_DISPLAY)
summary_source = pd.DataFrame({
    "malignant_state": display_state,
    "pseudotime": observed_pseudotime,
    "root_score": observed_root,
    "max_fate_probability": fate_df.max(axis=1).to_numpy(),
})
pseudotime_summary = (
    summary_source.groupby("malignant_state", observed=True)
    .agg(
        n_cells=("pseudotime", "size"),
        pseudotime_median=("pseudotime", "median"),
        pseudotime_mean=("pseudotime", "mean"),
        root_score_median=("root_score", "median"),
        root_score_mean=("root_score", "mean"),
        max_fate_probability_median=("max_fate_probability", "median"),
    ).reset_index()
)
mean_fate = fate_df.assign(malignant_state=display_state.to_numpy()).groupby(
    "malignant_state", observed=True
)[FATE_LABELS].mean().reset_index()

def terminal_display(value):
    value = str(value)
    if value in {"", "nan"}:
        return "Unassigned"
    if value.startswith("OPC_NPC_like_malignant"):
        return "OPC/NPC-like"
    return STATE_DISPLAY.get(value, value)

terminal_table = pd.crosstab(
    display_state,
    pd.Series([terminal_display(value) for value in observed_terminal], index=adata.obs_names),
).reindex(columns=["AC-like/reactive", "Hypoxia/stress", "NPC/neuronal-like", "OPC/NPC-like", "Unassigned"], fill_value=0).reset_index()

write_tsv(pseudotime_summary, METADATA_DIR / f"{OUT_PREFIX}_pseudotime_root_fate_summary_by_malignant_state.tsv")
write_tsv(mean_fate, METADATA_DIR / f"{OUT_PREFIX}_mean_fate_probability_by_malignant_state.tsv")
write_tsv(terminal_table, METADATA_DIR / f"{OUT_PREFIX}_terminal_state_counts_by_malignant_state.tsv")

# S10E: cell-wise maximum fate probability.
plot_umap(adata, MAX_FATE_PROBABILITY_KEY, "S10E_UMAP_maximum_fate_probability.png", "Maximum fate probability", size=5)

# S10F: maximum fate probability by malignant state.
plot_frame = pd.DataFrame({"malignant_state": display_state, "maximum_fate_probability": fate_df.max(axis=1).to_numpy()})
state_order = plot_frame.groupby("malignant_state")["maximum_fate_probability"].median().sort_values(ascending=False).index
values = [plot_frame.loc[plot_frame["malignant_state"] == state, "maximum_fate_probability"].to_numpy() for state in state_order]
fig, ax = plt.subplots(figsize=(10, 5.5))
ax.boxplot(values, tick_labels=state_order, showfliers=False, widths=0.55)
rng = np.random.default_rng(RANDOM_SEED)
for position, state_values in enumerate(values, start=1):
    ax.scatter(rng.normal(position, 0.045, len(state_values)), state_values, s=4, alpha=0.12, linewidths=0)
ax.set_ylabel("Maximum fate probability")
ax.set_xlabel("Malignant-state annotation")
ax.set_ylim(0, 1.02)
ax.tick_params(axis="x", rotation=35)
fig.tight_layout()
savefig(FIGDIR / "S10F_maximum_fate_probability_by_malignant_state.png")

# S10G: cell-wise fate probability toward each terminal macrostate.
for column in FATE_LABELS:
    safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", column).strip("_")
    key = f"fate_probability_{safe}"
    adata.obs[key] = fate_df[column].astype(float)
    plot_umap(adata, key, f"S10G_UMAP_{key}.png", column, size=5)


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\tables\cellrank_fate_probabilities.tsv (20000, 6)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_historical_checkpoint_audit.tsv (9, 4)


,checkpoint,expected,observed,match
0,selected_cells,20000,20000,True
1,state_counts,True,True,True
2,dpt_root_index,11977,11977,True
3,root_score_max_abs_diff,<=1e-06,0.0,True
4,pseudotime_max_abs_diff,<=1e-06,0.0,True
5,gpcca_result_source,historical_reference,historical_reference,True
6,terminal_state_cell_matches,20000,20000,True
7,fate_probability_max_abs_diff,<=1e-06,0.0,True
8,fate_lineages,6,6,True


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_pseudotime_root_fate_summary_by_malignant_state.tsv (5, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_mean_fate_probability_by_malignant_state.tsv (5, 7)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_terminal_state_counts_by_malignant_state.tsv (5, 6)
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\S10E_UMAP_maximum_fate_probability.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\S10F_maximum_fate_probability_by_malignant_state.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figures\S10G_UMAP_fate_probability_OPC_NPC-like.png
[FIG] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\figure

In [14]:
# Lineage-driver analysis was disabled in the historical S10 run.
lineage_driver_status = pd.DataFrame([
    {"item": "requested", "value": RUN_LINEAGE_DRIVERS},
    {"item": "executed", "value": False},
    {"item": "reason", "value": "Disabled historically and not required for Supplementary Figure S10"},
])
write_tsv(lineage_driver_status, METADATA_DIR / f"{OUT_PREFIX}_lineage_driver_status.tsv")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_lineage_driver_status.tsv (3, 2)


WindowsPath('G:/Repository/glioma-cnv-single-cell-atlas/metadata/CELLRANK_exploratory_S10_lineage_driver_status.tsv')

In [15]:
# Save final minimal analysis object and repository metadata
per_cell = adata.obs.copy()
for column in fate_df.columns:
    per_cell[f"fate_probability_{column}"] = fate_df[column].to_numpy()
write_tsv(per_cell.reset_index(), TABLEDIR / "cellrank_malignant_per_cell_metadata.tsv")

run_info.update({
    "finished_at": datetime.now().isoformat(timespec="seconds"),
    "final_n_cells": adata.n_obs,
    "final_expression_genes": adata.n_vars,
    "kernel_description": run_info["kernel_description"],
    "schur_method": SCHUR_METHOD,
    "historical_checkpoints_all_match": bool(historical_audit["match"].all()),
})
with open(LOGDIR / "cellrank_run_info.json", "w", encoding="utf-8") as handle:
    json.dump(run_info, handle, indent=2)

run_summary = pd.DataFrame([{"item": key, "value": value} for key, value in run_info.items()])
write_tsv(run_summary, METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv")

if SAVE_FINAL_H5AD:
    if FINAL_H5AD.exists():
        FINAL_H5AD.unlink()
    adata.write_h5ad(FINAL_H5AD, compression="gzip")
    print("[WRITE]", FINAL_H5AD)

manifest_paths = [
    HISTORICAL_REFERENCE,
    METADATA_DIR / f"{OUT_PREFIX}_readiness.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_memory_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_state_counts.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_historical_checkpoint_audit.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_current_latent_dpt_diagnostic.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_kernel_status.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_gpcca_status.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_pseudotime_root_fate_summary_by_malignant_state.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_mean_fate_probability_by_malignant_state.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_terminal_state_counts_by_malignant_state.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_lineage_driver_status.tsv",
    METADATA_DIR / f"{OUT_PREFIX}_run_summary.tsv",
    FINAL_H5AD,
]
manifest = pd.DataFrame([
    {"file": path.name, "path": str(path), "exists": path.exists(), "size_mb": round(path.stat().st_size / 1024**2, 3) if path.exists() else np.nan}
    for path in manifest_paths
])
write_tsv(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
if not manifest["exists"].all():
    raise AssertionError("One or more expected Notebook 31 outputs are missing.")

display(run_summary)
display(manifest)
gc.collect()
print("[DONE] Notebook 31 completed successfully.")


[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\tables\cellrank_malignant_per_cell_metadata.tsv (20000, 40)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_run_summary.tsv (28, 2)
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\results\cellrank_malignant_fate_mapping_v2_memory_safe\GBM_malignant_CellRank_PseudotimeKernel_result.h5ad
[WRITE] G:\Repository\glioma-cnv-single-cell-atlas\metadata\CELLRANK_exploratory_S10_output_manifest.tsv (14, 4)


,item,value
0,analysis,cellrank_malignant_fate_mapping_v2_memory_safe
1,started_at,2026-09-28T11:17:05
2,input_h5ad,G:\Repository\glioma-cnv-single-cell-atlas\dat...
3,historical_reference,G:\Repository\glioma-cnv-single-cell-atlas\met...
4,random_seed,13
5,use_rep,X_scVI
6,n_neighbors,30
7,n_schur_components,12
8,n_macrostates,6
9,pseudotime_kernel_weight,0.9


,file,path,exists,size_mb
0,CELLRANK_exploratory_S10_historical_reference....,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,2.021
1,CELLRANK_exploratory_S10_readiness.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
2,CELLRANK_exploratory_S10_memory_audit.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
3,CELLRANK_exploratory_S10_state_counts.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
4,CELLRANK_exploratory_S10_historical_checkpoint...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
5,CELLRANK_exploratory_S10_current_latent_dpt_di...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
6,CELLRANK_exploratory_S10_kernel_status.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
7,CELLRANK_exploratory_S10_gpcca_status.tsv,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000
8,CELLRANK_exploratory_S10_pseudotime_root_fate_...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001
9,CELLRANK_exploratory_S10_mean_fate_probability...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001


[DONE] Notebook 31 completed successfully.


## Supplementary Figure S10 output mapping

After successful execution, the `results/cellrank_malignant_fate_mapping_v2_memory_safe/figures/` directory contains:

- **S10A** — `UMAP_malignant_state.png`
- **S10B** — `UMAP_cr_dpt_pseudotime.png`
- **S10C** — `UMAP_cr_root_score.png`
- **S10D** — `CellRank_terminal_states.png`
- **S10E** — `S10E_UMAP_maximum_fate_probability.png`
- **S10F** — `S10F_maximum_fate_probability_by_malignant_state.png`
- **S10G** — per-lineage files beginning with `S10G_UMAP_fate_probability_`

The repository `metadata/` directory contains the historical checkpoint audit and the three numerical summaries used for the archived Supplementary Figure S10.
